# Fractals: Mandelbrot and Julia Sets
### Lesson 5, Section 3

Both the Mandelbrot set and Julia sets arise from a single iteration in the
complex plane:

$$z_{n+1} = z_n^2 + c$$

- **Mandelbrot set**: fix $z_0 = 0$, vary $c$. Which values of $c$ keep the
  iteration bounded?
- **Julia set**: fix $c$, vary $z_0$. Which starting points stay bounded?

You will implement the escape-time algorithm, render both sets, and explore
self-similarity through zooming.


## 0 · Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

print('All imports OK ✓')


---
# Part 3 — Fractals: Mandelbrot and Julia Sets

Both arise from one iteration in the complex plane:

$$z_{n+1} = z_n^2 + c$$

- **Mandelbrot set**: fix $z_0 = 0$, vary $c$. Which values of $c$ keep the iteration bounded?
- **Julia set**: fix $c$, vary $z_0$. Which starting points stay bounded?


## 3.1 · Iterate one point
We start by checking whether a single complex number $c$ belongs to the Mandelbrot set.


In [ ]:
def mandelbrot_escape(c, max_iter=100):
    """Return the escape iteration for a single complex number c.
    If the point does not escape within max_iter, return max_iter (= inside the set).
    """
    z = 0 + 0j
    for n in range(max_iter):
        z = z * z + c
        if abs(z) > 2:
            return n
    return max_iter

print('c = 0+0j  →', mandelbrot_escape(0 + 0j))
print('c = 1+0j  →', mandelbrot_escape(1 + 0j))
print('c = -1+0j →', mandelbrot_escape(-1 + 0j))


### Independent test 4
$c = 0$ is the centre of the Mandelbrot set and should never escape. $c = 2$ escapes immediately.


In [ ]:
# --- Test: known Mandelbrot membership ---
assert mandelbrot_escape(0 + 0j) == 100   # inside
assert mandelbrot_escape(2 + 0j) < 5      # escapes fast
assert mandelbrot_escape(-1 + 0j) == 100  # inside (period-2 point)
print('Test passed ✓ Known points classified correctly.')


## 3.2 · The Mandelbrot set — full image
We compute the escape time for a grid of complex numbers and visualise the result.


In [ ]:
def compute_mandelbrot(x_range=(-2, 1), y_range=(-1.5, 1.5), width=800, height=800, max_iter=100):
    """Compute escape-time array for the Mandelbrot set (vectorised)."""
    x = np.linspace(*x_range, width)
    y = np.linspace(*y_range, height)
    real, imag = np.meshgrid(x, y)
    c = real + 1j * imag
    z = np.zeros_like(c)
    escape_time = np.full(c.shape, max_iter, dtype=int)

    for i in range(max_iter):
        mask = np.abs(z) <= 2
        z[mask] = z[mask] ** 2 + c[mask]
        newly_escaped = (np.abs(z) > 2) & (escape_time == max_iter)
        escape_time[newly_escaped] = i

    return x, y, escape_time

x, y, escape = compute_mandelbrot()

fig, ax = plt.subplots(figsize=(9, 9))
im = ax.imshow(escape, extent=[x.min(), x.max(), y.min(), y.max()],
               origin='lower', cmap='inferno', aspect='equal')
ax.set_title('The Mandelbrot Set')
ax.set_xlabel('Re(c)')
ax.set_ylabel('Im(c)')
plt.colorbar(im, ax=ax, label='Escape iteration')
plt.tight_layout()
plt.show()


## 3.3 · Zooming into the Mandelbrot set
One of the most spectacular properties of fractals is that **zooming in reveals new detail** that resembles the whole.


In [ ]:
zoom_regions = [
    {'x_range': (-2, 1),       'y_range': (-1.5, 1.5),    'title': 'Full view'},
    {'x_range': (-0.8, -0.7),  'y_range': (0.1, 0.2),     'title': 'Zoom 1 — spiral'},
    {'x_range': (-0.748, -0.745), 'y_range': (0.1, 0.103), 'title': 'Zoom 2 — deeper'},
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, region in zip(axes, zoom_regions):
    x, y, esc = compute_mandelbrot(
        x_range=region['x_range'], y_range=region['y_range'],
        width=600, height=600, max_iter=300
    )
    ax.imshow(esc, extent=[x.min(), x.max(), y.min(), y.max()],
              origin='lower', cmap='inferno', aspect='equal')
    ax.set_title(region['title'])
    ax.set_xlabel('Re(c)')
    ax.set_ylabel('Im(c)')

plt.suptitle('Self-Similarity in the Mandelbrot Set', fontweight='bold')
plt.tight_layout()
plt.show()


## 3.4 · Julia sets
For a **fixed** $c$, a Julia set shows which initial values $z_0$ stay bounded.


In [ ]:
def compute_julia(c_value, x_range=(-2, 2), y_range=(-2, 2), width=800, height=800, max_iter=200):
    """Compute escape-time array for the Julia set of a given c."""
    x = np.linspace(*x_range, width)
    y = np.linspace(*y_range, height)
    real, imag = np.meshgrid(x, y)
    z = real + 1j * imag
    escape_time = np.full(z.shape, max_iter, dtype=int)

    for i in range(max_iter):
        mask = np.abs(z) <= 2
        z[mask] = z[mask] ** 2 + c_value
        newly_escaped = (np.abs(z) > 2) & (escape_time == max_iter)
        escape_time[newly_escaped] = i

    return x, y, escape_time


In [ ]:
c_values = [
    (-0.7 + 0.27015j, 'c = -0.70 + 0.27i'),
    (-0.4 + 0.6j,     'c = -0.40 + 0.60i'),
    (0.355 + 0.355j,  'c =  0.355 + 0.355i'),
    (-0.8 + 0.156j,   'c = -0.80 + 0.156i'),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for ax, (c_val, title) in zip(axes.flat, c_values):
    x, y, esc = compute_julia(c_val, max_iter=200)
    ax.imshow(esc, extent=[x.min(), x.max(), y.min(), y.max()],
              origin='lower', cmap='twilight_shifted', aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('Re($z_0$)')
    ax.set_ylabel('Im($z_0$)')

plt.suptitle('Julia Sets for Different Values of c', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()


### What to try
- Pick a $c$ **inside** the Mandelbrot set (e.g. $c = -0.1 + 0.7i$) and one **outside** (e.g. $c = 1$). Compare the Julia sets.
- Inside → **connected** fractal. Outside → **disconnected** dust.
- Increase `max_iter` for more detail near the boundary.
- Try `cmap='hot'`, `'ocean'`, `'cubehelix'` for different aesthetics.


---
## What to Try

1. Pick a $c$ **inside** the Mandelbrot set (e.g. $c = -0.1 + 0.7i$) and one
   **outside** (e.g. $c = 1$). Compare the Julia sets — inside gives a *connected*
   fractal; outside gives disconnected dust.
2. Pick a point on the boundary of the Mandelbrot set and zoom in 1000×.
   Describe what you see.
3. Animate a sequence of Julia sets where $c$ moves along a circle in the
   complex plane.
4. Increase `max_iter` — how does the boundary detail change?
5. Try `cmap='hot'`, `'ocean'`, `'cubehelix'` for different aesthetics.


---
## Mini-Projects

### Project C · Mandelbrot Set and the Logistic Map
Show that the logistic map $X_{N+1} = rX_N(1-X_N)$ can be transformed into
$z_{n+1} = z_n^2 + c$ with the substitution $z = r(\tfrac{1}{2} - X)$ and
$c = r/2 - r^2/4$. Overlay the real-axis slice of the Mandelbrot set on top
of the logistic-map bifurcation diagram to reveal the connection.

### Project D · Smooth Colouring
Render a high-resolution Mandelbrot set with **smooth colouring** using the
normalised iteration count technique:

$$\mu = n - \log_2 \log_2 |z_n|$$

This eliminates the 'banding' effect visible with integer escape times.
Experiment with custom colour maps and high iteration counts.

### Final Reflection
After completing the three sections, explain:
- Why deterministic systems can be unpredictable
- What the practical limits of prediction are for chaotic systems
- Where you encounter chaos and fractals in real-world phenomena
